In [29]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [30]:
df = pd.read_csv("data/animal-fun-facts-dataset.csv")
print(df.head())
print(df.columns)

  animal_name                                             source  \
0    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
1    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
2    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
3    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   
4    aardvark  https://www.animalfactsencyclopedia.com/Aardva...   

                                                text media_link  \
0  Aardvarks are sometimes called "ant bears", "e...        NaN   
1  Aardvarks\nhave rather primitive brains that a...        NaN   
2  Aardvarks\nteeth are lined with fine upright t...        NaN   
3  The aardvarks Latin family name "Tubulidentata...        NaN   
4  Baby aardvarks are born with front teeth that ...        NaN   

   wikipedia_link  
0  /wiki/Aardvark  
1  /wiki/Aardvark  
2  /wiki/Aardvark  
3  /wiki/Aardvark  
4  /wiki/Aardvark  
Index(['animal_name', 'source', 'text', 'media_link', 'wikipedia_lin

In [31]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documentos = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        textos = [doc.text for doc in documents]
        nuevos_embeddings = nuevos_embeddings = self.embedding_model.encode(textos)

        self.documentos.extend(documents)
        self.embeddings.extend(nuevos_embeddings)


    def similitud_coseno(self, vec_a, vec_b):
        producto_punto = np.dot(vec_a, vec_b)
        magnitud_a = np.linalg.norm(vec_a)
        magnitud_b = np.linalg.norm(vec_b)
        return producto_punto / (magnitud_a * magnitud_b)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embeddings = self.embedding_model.encode(query)
        resultados = []

        for i, emb in enumerate(self.embeddings):
            score = self.similitud_coseno(query_embeddings, emb)
            resultados.append((score, i))

        resultados_ordenados = sorted(resultados, key=lambda x: x[0], reverse=True)

        final = []
        for score, i in resultados_ordenados[:top_k]:
            final.append(SearchResult(score=score, document=self.documentos[i]))
        return final



In [32]:
documentos_animales = []

for index, row in df.iterrows():
    if pd.isna(row["text"]):
        continue

    doc = Document(
        text=str(row["text"]),
        metadata={
            "animal_name": str(row["animal_name"]) if not pd.isna(row["animal_name"]) else "",
            "source": str(row["source"]) if not pd.isna(row["source"]) else "",
            "media_link": str(row["media_link"]) if not pd.isna(row["media_link"]) else "",
            "wikipedia_link": str(row["wikipedia_link"]) if not pd.isna(row["wikipedia_link"]) else "",
        }
    )
    documentos_animales.append(doc)

print(f"Total documentos: {len(documentos_animales)}")

Total documentos: 7731


In [33]:
model = SentenceTransformer("all-MiniLM-L6-v2")
store = VectorStore(model)
store.add_documents(documentos_animales)

In [34]:
queries = [
    "animals that can fly",
    "dangerous and poisonous animals",
    "animals that live in the ocean",
    "fastest animals on land",
    "animals that sleep a lot"
]

for query in queries:
    print(f"\n=== Query: {query} ===")
    resultados = store.search(query, top_k=3)
    for r in resultados:
        print(f"Score: {r.score:.4f}")
        print(f"Texto: {r.document.text}")
        print(f"Metadata: {r.document.metadata}")
        print("---")

Score: 0.6895
Texto: They sleep during the day and are awake at night
Metadata: {'animal_name': 'syrian hamster', 'source': 'https://www.animalfactsencyclopedia.com/Syrian-hamster.html', 'media_link': '', 'wikipedia_link': '/wiki/Golden_hamster'}
---
Score: 0.6861
Texto: These animals are diurnal, sleeping in treetop leaves and branches during the night. They spend most of day in search of food, grooming, and resting.
Metadata: {'animal_name': 'coatimundi', 'source': 'https://seaworld.org/animals/facts/mammals/coatimundi/', 'media_link': '', 'wikipedia_link': '/wiki/Coati'}
---
Score: 0.6653
Texto: They often sleep 16 hours a day!.
In addition to being solitary animals, armadillos also like to sleep—a lot.
Metadata: {'animal_name': 'armadillo', 'source': 'https://factanimal.com/armadillo/', 'media_link': '', 'wikipedia_link': '/wiki/Armadillo'}
---


In [35]:


model = SentenceTransformer("all-MiniLM-L6-v2")
store = VectorStore(model)

docs = [
    Document("Los perros son leales y amigables", {"tipo": "animal"}),
    Document("El fútbol es el deporte más popular", {"tipo": "deporte"}),
    Document("Los gatos son independientes", {"tipo": "animal"}),
]

store.add_documents(docs)

resultados = store.search("¿Qué animal es buen compañero?", top_k=2)

for r in resultados:
    print(f"{r.score:.4f} — {r.document.text} — {r.document.metadata}")

0.4830 — Los perros son leales y amigables — {'tipo': 'animal'}
0.4743 — Los gatos son independientes — {'tipo': 'animal'}


In [36]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documentos = []
        self.embeddings = []


    def add_documents(self, documents: list[Document]):
        textos = [doc.text for doc in documents]
        nuevos_embeddings = nuevos_embeddings = self.embedding_model.encode(textos)

        self.documentos.extend(documents)
        self.embeddings.extend(nuevos_embeddings)


    def similitud_coseno(self, vec_a, vec_b):
        producto_punto = np.dot(vec_a, vec_b)
        magnitud_a = np.linalg.norm(vec_a)
        magnitud_b = np.linalg.norm(vec_b)
        return producto_punto / (magnitud_a * magnitud_b)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:

        query_embedding = self.embedding_model.encode(query)
        resultados = []

        if metadata_filter:
            indices_filtrados = [
                i for i, doc in enumerate(self.documentos)
                if all(doc.metadata.get(k) == v for k, v in metadata_filter.items())
            ]
        else:
            indices_filtrados = list(range(len(self.documentos)))

        for i in indices_filtrados:
            score = self.similitud_coseno(query_embedding, self.embeddings[i])
            resultados.append((score, i))

        resultados_ordenados = sorted(resultados, key=lambda x: x[0], reverse=True)

        final = []
        for score, i in resultados_ordenados[:top_k]:
            final.append(SearchResult(score=score, document=self.documentos[i]))
        return final

In [37]:
fstore = FilteredVectorStore(model)
fstore.add_documents(documentos_animales)

resultados = fstore.search(
    "fast and dangerous",
    top_k=3,
    metadata_filter={"animal_name": "cheetah"}
)

for r in resultados:
    print(f"Score: {r.score:.4f}")
    print(f"Texto: {r.document.text}")
    print(f"Metadata: {r.document.metadata}")

Score: 0.4209
Texto: The fastest land mammal in the world!
Metadata: {'animal_name': 'cheetah', 'source': 'https://a-z-animals.com/animals/cheetah/', 'media_link': '', 'wikipedia_link': '/wiki/Cheetah'}
Score: 0.3865
Texto: Cheetahs are the fastest land animal in the world..
These cats are fast. They can typically reach speeds of up to 98 kilometers per hour (61 miles per hour), and can go from 0 to 60 Mph in just 3-seconds, which is faster than most super-cars. Their stride length becomes as long as 7m (23ft) at full pace, which means the cheetah spends more than half the time airborne. 1
Metadata: {'animal_name': 'cheetah', 'source': 'https://factanimal.com/cheetahs/', 'media_link': '', 'wikipedia_link': '/wiki/Cheetah'}
Score: 0.3555
Texto: The average cheetahs top speed is probably about 65mph,
with some exceptional individuals reaching up to 70mph for no more than
3-4 seconds.
Metadata: {'animal_name': 'cheetah', 'source': 'https://www.animalfactsencyclopedia.com/Cheetah-facts.htm

In [38]:
df2 = pd.read_csv("data/IMDB_Top_250_Movies.csv")
print(df2.head())
print(df2.columns)
print(df2.shape)

   rank                      name  year  rating               genre  \
0     1  The Shawshank Redemption  1994     9.3               Drama   
1     2             The Godfather  1972     9.2         Crime,Drama   
2     3           The Dark Knight  2008     9.0  Action,Crime,Drama   
3     4     The Godfather Part II  1974     9.0         Crime,Drama   
4     5              12 Angry Men  1957     9.0         Crime,Drama   

  certificate run_time                                            tagline  \
0           R   2h 22m  Fear can hold you prisoner. Hope can set you f...   
1           R   2h 55m                         An offer you can't refuse.   
2       PG-13   2h 32m                                    Why So Serious?   
3           R   3h 22m       All the power on earth can't change destiny.   
4    Approved   1h 36m  Life Is In Their Hands -- Death Is On Their Mi...   

      budget  box_office                                              casts  \
0   25000000    28884504  Tim R

In [39]:
for index, row in df2.iterrows():
    if pd.isna(row["tagline"]):
        continue
    doc = Document(
        text=row["tagline"],
        metadata={
            "name": str(row["name"]) if not pd.isna(row["name"]) else "",
            "year": str(row["year"]) if not pd.isna(row["year"]) else "",
            "rating": str(row["rating"]) if not pd.isna(row["rating"]) else "",
            "genre": str(row["genre"]) if not pd.isna(row["genre"]) else "",
            "directors": str(row["directors"]) if not pd.isna(row["directors"]) else ""
        }
    )

In [40]:
fstore2 = FilteredVectorStore(model)
fstore2.add_documents(documentos_peliculas)

In [41]:
queries_peliculas = [
    ("movies about hope and freedom", {"genre": "Drama"}),
    ("story about crime and power", {"directors": "Francis Ford Coppola"}),
    ("superhero fighting evil", {"genre": "Action,Crime,Drama"}),
    ("war and sacrifice", {"genre": "Drama"}),
    ("love and destiny", {"directors": "Christopher Nolan"}),
]

for query, filtro in queries_peliculas:
    print(f"\nQuery: {query} | Filtro: {filtro}")
    resultados = fstore2.search(query, top_k=3, metadata_filter=filtro)
    for r in resultados:
        print(f"Score: {r.score:.4f}")
        print(f"Texto: {r.document.text}")
        print(f"Metadata: {r.document.metadata}")


Query: movies about hope and freedom | Filtro: {'genre': 'Drama'}
Score: 0.4414
Texto: Fear can hold you prisoner. Hope can set you free.
Metadata: {'name': 'The Shawshank Redemption', 'year': '1994', 'rating': '9.3', 'genre': 'Drama', 'directors': 'Frank Darabont'}
Score: 0.4383
Texto: One of the Great Films of Our Time!
Metadata: {'name': 'Ikiru', 'year': '1952', 'rating': '8.3', 'genre': 'Drama', 'directors': 'Akira Kurosawa'}
Score: 0.3663
Texto: "NETWORK"... the humanoids, the love story, the trials and tribulations, the savior of television, the attempted suicides, the assassination -- it's ALL coming along with a galaxy of stars you know and love!
Metadata: {'name': 'Network', 'year': '1976', 'rating': '8.1', 'genre': 'Drama', 'directors': 'Sidney Lumet'}

Query: story about crime and power | Filtro: {'directors': 'Francis Ford Coppola'}
Score: 0.2404
Texto: All the power on earth can't change destiny.
Metadata: {'name': 'The Godfather Part II', 'year': '1974', 'rating': '9.0',

En conclusión esta actividad me ayudó a comprender mejor el funcionamiento de un vector store, comprendiendo la manera que se tienen que almacenar los embeddings para que posteriormente se hagan las solicitudes de querys, se realizó además la implementación de la similitud coseno para observar la relación que existe entre las solicitudes y la lista de documentos, marcando el top 5 que tenían una mayor similitud y eran relevantes.